In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve

pd.options.display.float_format = "{:.3f}".format

SCORE_DECIMALS = 3


def round_float_columns(df: pd.DataFrame, decimals: int = SCORE_DECIMALS) -> pd.DataFrame:
    rounded = df.copy()
    float_cols = rounded.select_dtypes(include=["float", "float64", "float32"]).columns
    rounded[float_cols] = rounded[float_cols].round(decimals)
    return rounded


In [ ]:
# 프로젝트 경로 설정
PROJECT_DIR = Path.cwd().resolve().parent
DATA_SPLIT_DIR = PROJECT_DIR / "processed" / "data_split"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODELING_OUTPUT_DIR = OUTPUT_DIR / "modeling"
WITHIN_OUTPUT_DIR = MODELING_OUTPUT_DIR / "within_t_plus_2"
FIGURE_DIR = MODELING_OUTPUT_DIR / "figures"
RESULT_ANALYSIS_DIR = OUTPUT_DIR / "result_analysis"
RESULT_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_SPLIT_DIR:", DATA_SPLIT_DIR)
print("MODELING_OUTPUT_DIR:", MODELING_OUTPUT_DIR)
print("WITHIN_OUTPUT_DIR:", WITHIN_OUTPUT_DIR)
print("FIGURE_DIR:", FIGURE_DIR)
print("RESULT_ANALYSIS_DIR:", RESULT_ANALYSIS_DIR)


In [ ]:

# Analysis settings
THRESHOLD = 0.5
MODEL_PROB_COLUMNS = {
    "LR": "lr_prob",
    "RF": "rf_prob",
    "XGB": "xgb_prob",
    "LightGBM": "lgbm_prob",
    "MLP": "mlp_prob",
    "Single-output LSTM": "lstm_prob",
}
HORIZONS = ["y_t", "y_t_plus_1", "y_t_plus_2"]
HORIZON_LABELS = {"y_t": "t", "y_t_plus_1": "t+1", "y_t_plus_2": "t+2"}

BOOTSTRAP_N_ITER = 4000
BOOTSTRAP_RANDOM_SEED = 42
BOOTSTRAP_METRICS = ["auroc", "auprc"]
BOOTSTRAP_METRIC_LABELS = {"auroc": "AUROC", "auprc": "AUPRC"}
BOOTSTRAP_Y_LIMITS = {"auroc": (0.6, 0.9), "auprc": (0.5, 0.8)}
MODEL_COLORS = {
    "LR": "#4c78a8",
    "RF": "#f58518",
    "XGB": "#54a24b",
    "LightGBM": "#b279a2",
    "MLP": "#e45756",
    "Single-output LSTM": "#72b7b2",
    "LSTM": "#72b7b2",
}


## Load modeling outputs

In [ ]:

# Load modeling outputs
binned = pd.read_csv(DATA_SPLIT_DIR / "events_12h_binned_with_split.csv")
within_summary = pd.read_csv(WITHIN_OUTPUT_DIR / "within_t_plus_2_test_metrics_summary.csv")
within_predictions = pd.read_csv(WITHIN_OUTPUT_DIR / "within_t_plus_2_test_predictions_all_models.csv")

multi_summary_raw = pd.read_csv(MODELING_OUTPUT_DIR / "multi_horizon_test_metrics_summary.csv")
lgbm_multi_summary = pd.read_csv(MODELING_OUTPUT_DIR / "lgbm_multi_horizon_test_metrics.csv")
lgbm_multi_predictions = pd.read_csv(MODELING_OUTPUT_DIR / "lgbm_multi_horizon_test_predictions.csv")
mlp_multi_predictions = pd.read_csv(MODELING_OUTPUT_DIR / "mlp_multi_horizon_test_predictions.csv")
multi_predictions = pd.read_csv(MODELING_OUTPUT_DIR / "lstm_gpu_test_predictions.csv")

lgbm_multi_horizon_metrics = pd.read_csv(MODELING_OUTPUT_DIR / "lgbm_multi_horizon_test_metrics_by_horizon.csv")
mlp_multi_horizon_metrics = pd.read_csv(MODELING_OUTPUT_DIR / "mlp_multi_horizon_test_metrics_by_horizon.csv")
multi_horizon_metrics = pd.read_csv(MODELING_OUTPUT_DIR / "lstm_gpu_test_metrics_by_horizon.csv")
multi_test_metrics = pd.read_csv(MODELING_OUTPUT_DIR / "lstm_gpu_test_metrics.csv")

multi_summary = pd.concat(
    [
        lgbm_multi_summary,
        multi_summary_raw[multi_summary_raw["model"].isin(["MLP_multi_horizon", "LSTM_encoder_decoder"])],
    ],
    ignore_index=True,
    sort=False,
)

within_summary = within_summary.assign(
    positive_rate=(within_summary["tp"] + within_summary["fn"])
    / (within_summary["tp"] + within_summary["fp"] + within_summary["tn"] + within_summary["fn"])
)

within_core_metrics = (
    within_summary[["model", "positive_rate", "auroc", "auprc", "sensitivity", "specificity", "ppv", "npv"]]
    .rename(
        columns={
            "positive_rate": "Positive_rate",
            "auroc": "AUROC",
            "auprc": "AUPRC",
            "sensitivity": "Sensitivity",
            "specificity": "Specificity",
            "ppv": "PPV",
            "npv": "NPV",
        }
    )
    .sort_values("AUPRC", ascending=False)
)

multi_prediction_sets = [
    ("LGBM_multi_horizon", lgbm_multi_predictions),
    ("MLP_multi_horizon", mlp_multi_predictions),
    ("LSTM_encoder_decoder", multi_predictions),
]

multi_prediction_lookup = dict(multi_prediction_sets)
multi_core_rows = []
for _, summary_row in multi_summary.iterrows():
    model = summary_row["model"]
    prediction_df = multi_prediction_lookup[model]
    for horizon, horizon_label in zip(HORIZONS, ["t", "t+1", "t+2"]):
        evaluable = prediction_df[prediction_df[f"{horizon}_mask"].eq(1)]
        multi_core_rows.append(
            {
                "model": model,
                "horizon": horizon_label,
                "Positive_rate": evaluable[f"{horizon}_true"].mean(),
                "AUROC": summary_row[f"{horizon}_auroc"],
                "AUPRC": summary_row[f"{horizon}_auprc"],
                "n_evaluable": summary_row[f"{horizon}_n_evaluable"],
            }
        )

multi_core_metrics = pd.DataFrame(multi_core_rows)
multi_core_metrics["horizon"] = pd.Categorical(multi_core_metrics["horizon"], categories=["t", "t+1", "t+2"], ordered=True)
multi_core_metrics = multi_core_metrics.sort_values(["horizon", "AUPRC"], ascending=[True, False]).reset_index(drop=True)

print("within_predictions", within_predictions.shape)
print("lgbm_multi_predictions", lgbm_multi_predictions.shape)
print("mlp_multi_predictions", mlp_multi_predictions.shape)
print("multi_predictions", multi_predictions.shape)

print("Within t+2 core test metrics")
display(round_float_columns(within_core_metrics))
print("Multi-horizon core test metrics")
display(round_float_columns(multi_core_metrics))


## Metric functions

In [ ]:

def likelihood_ratio(numerator: float, denominator: float) -> float:
    if pd.isna(numerator) or pd.isna(denominator):
        return np.nan
    if denominator == 0:
        return np.inf if numerator > 0 else np.nan
    return numerator / denominator


def binary_metric_dict(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = THRESHOLD) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    if len(y_true) == 0:
        return {
            "n": 0,
            "positive_rate": np.nan,
            "auroc": np.nan,
            "auprc": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "ppv": np.nan,
            "npv": np.nan,
            "lr_positive": np.nan,
            "lr_negative": np.nan,
            "tp": 0,
            "fp": 0,
            "tn": 0,
            "fn": 0,
        }

    if len(np.unique(y_true)) == 2:
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
    else:
        auroc = np.nan
        auprc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    lr_positive = likelihood_ratio(sensitivity, 1 - specificity)
    lr_negative = likelihood_ratio(1 - sensitivity, specificity)

    return {
        "n": int(len(y_true)),
        "positive_rate": float(y_true.mean()),
        "auroc": auroc,
        "auprc": auprc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "lr_positive": lr_positive,
        "lr_negative": lr_negative,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def add_metric_row(rows: list[dict], analysis: str, model: str, group: str, target: str, y_true, y_prob):
    row = {
        "analysis": analysis,
        "model": model,
        "prev_delirium_group": group,
        "target": target,
        **binary_metric_dict(y_true, y_prob),
    }
    rows.append(row)


CORE_METRIC_COLUMNS = ["auroc", "auprc", "sensitivity", "specificity", "ppv", "npv"]
CORE_METRIC_RENAME = {
    "auroc": "AUROC",
    "auprc": "AUPRC",
    "sensitivity": "Sensitivity",
    "specificity": "Specificity",
    "ppv": "PPV",
    "npv": "NPV",
}
PREV_DELIRIUM_METRIC_COLUMNS = [*CORE_METRIC_COLUMNS, "lr_positive", "lr_negative"]
PREV_DELIRIUM_METRIC_RENAME = {
    **CORE_METRIC_RENAME,
    "lr_positive": "LR+",
    "lr_negative": "LR-",
}


def save_key_result(df: pd.DataFrame, filename: str) -> Path:
    output_path = RESULT_ANALYSIS_DIR / filename
    round_float_columns(df.replace([np.inf, -np.inf], np.nan)).to_csv(output_path, index=False, float_format=f"%.{SCORE_DECIMALS}f")
    return output_path


def core_metric_table(
    metrics_df: pd.DataFrame,
    leading_cols: list[str] | tuple[str, ...] = ("model",),
    include_counts: bool = False,
    sort_by: str = "AUPRC",
) -> pd.DataFrame:
    cols = [*leading_cols, *CORE_METRIC_COLUMNS]
    if include_counts:
        cols.extend(["n", "positive_rate"])
    table = metrics_df[cols].rename(columns=CORE_METRIC_RENAME)
    if sort_by in table.columns:
        table = table.sort_values(sort_by, ascending=False)
    return table.reset_index(drop=True)


def metric_value(y_true, y_prob, metric: str) -> float:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return np.nan
    if metric == "auroc":
        return roc_auc_score(y_true, y_prob)
    if metric == "auprc":
        return average_precision_score(y_true, y_prob)
    raise ValueError(f"Unsupported metric: {metric}")


def bootstrap_metric_distribution(
    y_true,
    y_prob,
    metrics: list[str] = BOOTSTRAP_METRICS,
    n_bootstrap: int = BOOTSTRAP_N_ITER,
    seed: int = BOOTSTRAP_RANDOM_SEED,
) -> pd.DataFrame:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    rng = np.random.default_rng(seed)
    rows = []

    for iteration in range(n_bootstrap):
        sample_idx = rng.integers(0, len(y_true), size=len(y_true))
        sampled_true = y_true[sample_idx]
        sampled_prob = y_prob[sample_idx]
        for metric in metrics:
            rows.append(
                {
                    "bootstrap_iteration": iteration,
                    "metric": metric,
                    "value": metric_value(sampled_true, sampled_prob, metric),
                }
            )
    return pd.DataFrame(rows).dropna(subset=["value"])


def paired_bootstrap_metric_distribution(
    y_true,
    single_prob,
    multi_prob,
    metrics: list[str] = BOOTSTRAP_METRICS,
    n_bootstrap: int = BOOTSTRAP_N_ITER,
    seed: int = BOOTSTRAP_RANDOM_SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    y_true = np.asarray(y_true).astype(int)
    single_prob = np.asarray(single_prob).astype(float)
    multi_prob = np.asarray(multi_prob).astype(float)
    rng = np.random.default_rng(seed)
    distribution_rows = []
    difference_rows = []

    for iteration in range(n_bootstrap):
        sample_idx = rng.integers(0, len(y_true), size=len(y_true))
        sampled_true = y_true[sample_idx]
        sampled_single = single_prob[sample_idx]
        sampled_multi = multi_prob[sample_idx]

        for metric in metrics:
            single_value = metric_value(sampled_true, sampled_single, metric)
            multi_value = metric_value(sampled_true, sampled_multi, metric)
            if pd.isna(single_value) or pd.isna(multi_value):
                continue
            distribution_rows.extend(
                [
                    {
                        "bootstrap_iteration": iteration,
                        "metric": metric,
                        "variant": "Single output",
                        "value": single_value,
                    },
                    {
                        "bootstrap_iteration": iteration,
                        "metric": metric,
                        "variant": "Multi-horizon",
                        "value": multi_value,
                    },
                ]
            )
            difference_rows.append(
                {
                    "bootstrap_iteration": iteration,
                    "metric": metric,
                    "delta_multi_minus_single": multi_value - single_value,
                }
            )

    return pd.DataFrame(distribution_rows), pd.DataFrame(difference_rows)


def paired_bootstrap_p_value(delta: pd.Series) -> float:
    delta = delta.dropna()
    if delta.empty:
        return np.nan
    p_value = 2 * min((delta <= 0).mean(), (delta >= 0).mean())
    return min(float(p_value), 1.0)


def format_p_value(p_value: float) -> str:
    if pd.isna(p_value):
        return "p=NA"
    if p_value < 0.001:
        return "p<0.001"
    return f"p={p_value:.3f}"



def bootstrap_cache_is_valid(
    cache_df: pd.DataFrame,
    expected_models: list[str],
    expected_variants: list[str] | None = None,
    expected_metrics: list[str] = BOOTSTRAP_METRICS,
) -> bool:
    required_cols = {"bootstrap_iteration", "metric", "model", "value"}
    if expected_variants is not None:
        required_cols.add("variant")
    if not required_cols.issubset(cache_df.columns):
        return False
    if set(cache_df["model"].dropna().unique()) != set(expected_models):
        return False
    if set(cache_df["metric"].dropna().unique()) != set(expected_metrics):
        return False
    if expected_variants is not None and set(cache_df["variant"].dropna().unique()) != set(expected_variants):
        return False
    if cache_df["bootstrap_iteration"].nunique() != BOOTSTRAP_N_ITER:
        return False

    group_cols = ["model", "metric"] + (["variant"] if expected_variants is not None else [])
    expected_group_count = BOOTSTRAP_N_ITER
    return bool(cache_df.groupby(group_cols)["value"].count().ge(expected_group_count).all())


def load_bootstrap_cache(
    cache_path: Path,
    expected_models: list[str],
    expected_variants: list[str] | None = None,
) -> pd.DataFrame | None:
    if not cache_path.exists():
        return None
    cache_df = pd.read_csv(cache_path)
    if bootstrap_cache_is_valid(cache_df, expected_models=expected_models, expected_variants=expected_variants):
        print(f"Loaded cached bootstrap distributions from {cache_path.name}")
        return cache_df
    print(f"Ignoring stale bootstrap cache: {cache_path.name}")
    return None


def normalize_p_value_metric_names(p_value_df: pd.DataFrame) -> pd.DataFrame:
    p_value_df = p_value_df.copy()
    inverse_metric_labels = {label: metric for metric, label in BOOTSTRAP_METRIC_LABELS.items()}
    p_value_df["metric"] = p_value_df["metric"].replace(inverse_metric_labels)
    return p_value_df


def p_value_cache_is_valid(
    p_value_df: pd.DataFrame,
    expected_models: list[str],
    expected_metrics: list[str] = BOOTSTRAP_METRICS,
) -> bool:
    required_cols = {"model", "metric", "p_value"}
    if not required_cols.issubset(p_value_df.columns):
        return False
    if set(p_value_df["model"].dropna().unique()) != set(expected_models):
        return False
    if set(p_value_df["metric"].dropna().unique()) != set(expected_metrics):
        return False
    expected_pairs = len(expected_models) * len(expected_metrics)
    return len(p_value_df[["model", "metric"]].drop_duplicates()) == expected_pairs


def load_p_value_cache(cache_path: Path, expected_models: list[str]) -> pd.DataFrame | None:
    if not cache_path.exists():
        return None
    p_value_df = normalize_p_value_metric_names(pd.read_csv(cache_path))
    if p_value_cache_is_valid(p_value_df, expected_models=expected_models):
        print(f"Loaded cached bootstrap p-values from {cache_path.name}")
        return p_value_df
    print(f"Ignoring stale bootstrap p-value cache: {cache_path.name}")
    return None


## Overall result visualization

In [ ]:

# Within t+2 model-level AUROC / AUPRC curves and bootstrap box plots

def plot_roc_pr_curves(prediction_df: pd.DataFrame, target_col: str, model_prob_columns: dict[str, str], title_prefix: str, output_path: Path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    y_true_all = prediction_df[target_col].astype(int)
    prevalence = y_true_all.mean()

    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Chance")
    axes[1].axhline(prevalence, linestyle="--", color="gray", linewidth=1, label=f"Prevalence = {prevalence:.3f}")

    for model, prob_col in model_prob_columns.items():
        curve_df = prediction_df[[target_col, prob_col]].dropna()
        if curve_df[target_col].nunique() < 2:
            continue

        y_true = curve_df[target_col].astype(int)
        y_prob = curve_df[prob_col].astype(float)

        fpr, tpr, _ = roc_curve(y_true, y_prob)
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)

        axes[0].plot(fpr, tpr, linewidth=2, label=f"{model} (AUROC {auroc:.3f})")
        axes[1].plot(recall, precision, linewidth=2, label=f"{model} (AUPRC {auprc:.3f})")

    axes[0].set_xlabel("False positive rate")
    axes[0].set_ylabel("True positive rate")
    axes[0].set_title(f"{title_prefix}: ROC curve")
    axes[0].set_xlim(0, 1)
    axes[0].set_ylim(0, 1)
    axes[0].grid(alpha=0.3)
    axes[0].legend(frameon=False, fontsize=9)

    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title(f"{title_prefix}: Precision-Recall curve")
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[1].grid(alpha=0.3)
    axes[1].legend(frameon=False, fontsize=9)

    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()


def build_within_bootstrap_distributions(
    prediction_df: pd.DataFrame,
    target_col: str,
    model_prob_columns: dict[str, str],
    base_seed: int = BOOTSTRAP_RANDOM_SEED,
) -> pd.DataFrame:
    rows = []
    for model_idx, (model, prob_col) in enumerate(model_prob_columns.items()):
        metric_df = prediction_df[[target_col, prob_col]].dropna()
        distribution = bootstrap_metric_distribution(
            metric_df[target_col],
            metric_df[prob_col],
            seed=base_seed + model_idx,
        )
        distribution["model"] = model
        rows.append(distribution)
    return pd.concat(rows, ignore_index=True)


def plot_metric_boxplots_by_model(bootstrap_df: pd.DataFrame, title_prefix: str, output_path: Path):
    models = list(dict.fromkeys(bootstrap_df["model"]))
    fig, axes = plt.subplots(1, len(BOOTSTRAP_METRICS), figsize=(5.2 * len(BOOTSTRAP_METRICS), 5), sharey=False)
    if len(BOOTSTRAP_METRICS) == 1:
        axes = [axes]

    for ax, metric in zip(axes, BOOTSTRAP_METRICS):
        metric_df = bootstrap_df[bootstrap_df["metric"].eq(metric)]
        data = [metric_df.loc[metric_df["model"].eq(model), "value"].dropna().values for model in models]
        box = ax.boxplot(data, tick_labels=models, showfliers=False, patch_artist=True, medianprops={"color": "black"})
        for patch, model in zip(box["boxes"], models):
            patch.set_facecolor(MODEL_COLORS.get(model, "#8f8f8f"))
            patch.set_alpha(0.85)
        ax.set_title(f"{title_prefix}: {BOOTSTRAP_METRIC_LABELS[metric]}")
        ax.set_ylabel("Bootstrap score")
        ax.set_ylim(*BOOTSTRAP_Y_LIMITS[metric])
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", rotation=35)

    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()


plot_roc_pr_curves(
    within_predictions,
    target_col="y_within_t_plus_2_true",
    model_prob_columns=MODEL_PROB_COLUMNS,
    title_prefix="Within t+2 test",
    output_path=FIGURE_DIR / "within_t_plus_2_test_roc_pr_curves_by_model.png",
)

within_t2_bootstrap_cache_path = MODELING_OUTPUT_DIR / "within_t_plus_2_bootstrap_metric_distributions.csv"
within_t2_bootstrap = load_bootstrap_cache(
    within_t2_bootstrap_cache_path,
    expected_models=list(MODEL_PROB_COLUMNS.keys()),
)
if within_t2_bootstrap is None:
    within_t2_bootstrap = build_within_bootstrap_distributions(
        within_predictions,
        target_col="y_within_t_plus_2_true",
        model_prob_columns=MODEL_PROB_COLUMNS,
    )
    within_t2_bootstrap.to_csv(within_t2_bootstrap_cache_path, index=False)
    print(f"Saved bootstrap distributions to {within_t2_bootstrap_cache_path.name}")

plot_metric_boxplots_by_model(
    within_t2_bootstrap,
    title_prefix="Within t+2 test bootstrap",
    output_path=FIGURE_DIR / "within_t_plus_2_bootstrap_metric_boxplots_by_model.png",
)


In [ ]:

# Single-output vs multi-horizon within t+2 bootstrap box plots and LightGBM horizon curves
multi_within_curve_df = pd.DataFrame(
    {
        "y_within_t_plus_2_true": lgbm_multi_predictions["y_within_t_plus_2_true"].astype(int),
        "lgbm_prob": lgbm_multi_predictions["y_within_t_plus_2_prob_from_horizons"],
        "mlp_prob": mlp_multi_predictions["y_within_t_plus_2_prob_from_horizons"],
        "lstm_prob": multi_predictions["y_within_t_plus_2_prob_from_horizons"],
    }
)

plot_roc_pr_curves(
    multi_within_curve_df,
    target_col="y_within_t_plus_2_true",
    model_prob_columns={
        "LightGBM independent horizons": "lgbm_prob",
        "MLP multi-output": "mlp_prob",
        "LSTM encoder-decoder": "lstm_prob",
    },
    title_prefix="Multi-horizon within t+2 test",
    output_path=FIGURE_DIR / "multi_horizon_within_t_plus_2_roc_pr_curves_by_model.png",
)

single_multi_specs = [
    {
        "model": "LightGBM",
        "single_df": within_predictions,
        "single_prob_col": "lgbm_prob",
        "multi_df": lgbm_multi_predictions,
        "multi_prob_col": "y_within_t_plus_2_prob_from_horizons",
    },
    {
        "model": "MLP",
        "single_df": within_predictions,
        "single_prob_col": "mlp_prob",
        "multi_df": mlp_multi_predictions,
        "multi_prob_col": "y_within_t_plus_2_prob_from_horizons",
    },
    {
        "model": "LSTM",
        "single_df": within_predictions,
        "single_prob_col": "lstm_prob",
        "multi_df": multi_predictions,
        "multi_prob_col": "y_within_t_plus_2_prob_from_horizons",
    },
]


def paired_within_t2_frame(spec: dict) -> pd.DataFrame:
    id_cols = ["example_id", "stay_id", "anchor_bin"]
    single_df = spec["single_df"][[*id_cols, "y_within_t_plus_2_true", spec["single_prob_col"]]].rename(
        columns={
            "y_within_t_plus_2_true": "single_true",
            spec["single_prob_col"]: "single_prob",
        }
    )
    multi_df = spec["multi_df"][[*id_cols, "y_within_t_plus_2_true", spec["multi_prob_col"]]].rename(
        columns={
            "y_within_t_plus_2_true": "multi_true",
            spec["multi_prob_col"]: "multi_prob",
        }
    )
    paired = single_df.merge(multi_df, on=id_cols, how="inner").dropna(subset=["single_true", "multi_true", "single_prob", "multi_prob"])
    if not paired["single_true"].astype(int).equals(paired["multi_true"].astype(int)):
        raise ValueError(f"Mismatched within t+2 labels for {spec['model']}")
    paired["y_true"] = paired["single_true"].astype(int)
    return paired


single_vs_multi_models = [spec["model"] for spec in single_multi_specs]
single_vs_multi_bootstrap_cache_path = MODELING_OUTPUT_DIR / "single_vs_multi_horizon_within_t_plus_2_bootstrap_metric_distributions.csv"
single_vs_multi_p_values_cache_path = MODELING_OUTPUT_DIR / "single_vs_multi_horizon_within_t_plus_2_bootstrap_p_values.csv"

single_vs_multi_bootstrap = load_bootstrap_cache(
    single_vs_multi_bootstrap_cache_path,
    expected_models=single_vs_multi_models,
    expected_variants=["Single output", "Multi-horizon"],
)
single_vs_multi_p_values = load_p_value_cache(
    single_vs_multi_p_values_cache_path,
    expected_models=single_vs_multi_models,
)

if single_vs_multi_bootstrap is None or single_vs_multi_p_values is None:
    single_multi_bootstrap_rows = []
    single_multi_p_value_rows = []
    for spec_idx, spec in enumerate(single_multi_specs):
        paired = paired_within_t2_frame(spec)
        distribution, differences = paired_bootstrap_metric_distribution(
            paired["y_true"],
            paired["single_prob"],
            paired["multi_prob"],
            seed=BOOTSTRAP_RANDOM_SEED + 100 + spec_idx,
        )
        distribution["model"] = spec["model"]
        single_multi_bootstrap_rows.append(distribution)

        for metric in BOOTSTRAP_METRICS:
            metric_delta = differences.loc[differences["metric"].eq(metric), "delta_multi_minus_single"]
            single_multi_p_value_rows.append(
                {
                    "model": spec["model"],
                    "metric": metric,
                    "single_output_observed": metric_value(paired["y_true"], paired["single_prob"], metric),
                    "multi_horizon_observed": metric_value(paired["y_true"], paired["multi_prob"], metric),
                    "delta_multi_minus_single_observed": metric_value(paired["y_true"], paired["multi_prob"], metric) - metric_value(paired["y_true"], paired["single_prob"], metric),
                    "bootstrap_delta_mean": metric_delta.mean(),
                    "bootstrap_delta_ci_low": metric_delta.quantile(0.025),
                    "bootstrap_delta_ci_high": metric_delta.quantile(0.975),
                    "p_value": paired_bootstrap_p_value(metric_delta),
                }
            )

    single_vs_multi_bootstrap = pd.concat(single_multi_bootstrap_rows, ignore_index=True)
    single_vs_multi_bootstrap.to_csv(single_vs_multi_bootstrap_cache_path, index=False)

    single_vs_multi_p_values = pd.DataFrame(single_multi_p_value_rows)
    single_vs_multi_p_values.to_csv(single_vs_multi_p_values_cache_path, index=False)
    print(f"Saved bootstrap distributions to {single_vs_multi_bootstrap_cache_path.name}")
    print(f"Saved bootstrap p-values to {single_vs_multi_p_values_cache_path.name}")

single_vs_multi_p_values_display = single_vs_multi_p_values.copy()
single_vs_multi_p_values_display["metric"] = single_vs_multi_p_values_display["metric"].map(BOOTSTRAP_METRIC_LABELS)
display(round_float_columns(single_vs_multi_p_values_display))

def plot_single_vs_multi_bootstrap_boxplots(bootstrap_df: pd.DataFrame, p_value_df: pd.DataFrame, output_path: Path):
    models = ["LightGBM", "MLP", "LSTM"]
    variants = ["Single output", "Multi-horizon"]
    variant_alpha = {"Single output": 0.45, "Multi-horizon": 0.9}
    variant_hatch = {"Single output": "//", "Multi-horizon": ""}
    fig, axes = plt.subplots(1, len(BOOTSTRAP_METRICS), figsize=(5.8 * len(BOOTSTRAP_METRICS), 5), sharey=False)
    if len(BOOTSTRAP_METRICS) == 1:
        axes = [axes]

    for ax, metric in zip(axes, BOOTSTRAP_METRICS):
        metric_df = bootstrap_df[bootstrap_df["metric"].eq(metric)]
        positions = []
        data = []
        box_styles = []
        for model_idx, model in enumerate(models, start=1):
            for offset, variant in [(-0.18, "Single output"), (0.18, "Multi-horizon")]:
                positions.append(model_idx + offset)
                data.append(metric_df.loc[(metric_df["model"].eq(model)) & (metric_df["variant"].eq(variant)), "value"].dropna().values)
                box_styles.append((MODEL_COLORS.get(model, "#8f8f8f"), variant_alpha[variant], variant_hatch[variant]))

        box = ax.boxplot(data, positions=positions, widths=0.28, showfliers=False, patch_artist=True, medianprops={"color": "black"})
        for patch, (color, alpha, hatch) in zip(box["boxes"], box_styles):
            patch.set_facecolor(color)
            patch.set_alpha(alpha)
            patch.set_hatch(hatch)

        for model_idx, model in enumerate(models, start=1):
            p_value = p_value_df.loc[(p_value_df["model"].eq(model)) & (p_value_df["metric"].eq(metric)), "p_value"].iloc[0]
            model_values = metric_df.loc[metric_df["model"].eq(model), "value"]
            y_min, y_max = BOOTSTRAP_Y_LIMITS[metric]
            y_pos = min(y_max - 0.025, max(y_min + 0.04, model_values.quantile(0.995) + 0.01))
            ax.plot([model_idx - 0.18, model_idx + 0.18], [y_pos, y_pos], color="black", linewidth=1)
            ax.text(model_idx, y_pos + 0.006, format_p_value(p_value), ha="center", va="bottom", fontsize=9)

        ax.set_xticks(range(1, len(models) + 1))
        ax.set_xticklabels(models)
        ax.set_title(f"Single vs multi-horizon within t+2: {BOOTSTRAP_METRIC_LABELS[metric]}")
        ax.set_ylabel("Bootstrap score")
        ax.set_ylim(*BOOTSTRAP_Y_LIMITS[metric])
        ax.grid(axis="y", alpha=0.3)
        handles = [
            plt.Rectangle((0, 0), 1, 1, facecolor="#777777", alpha=variant_alpha[variant], hatch=variant_hatch[variant], label=variant)
            for variant in variants
        ]
        ax.legend(handles=handles, frameon=False, fontsize=9, loc="lower right")

    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()


plot_single_vs_multi_bootstrap_boxplots(
    single_vs_multi_bootstrap,
    single_vs_multi_p_values,
    output_path=FIGURE_DIR / "single_vs_multi_horizon_within_t_plus_2_bootstrap_boxplots.png",
)


def plot_lgbm_multi_horizon_roc_pr_by_horizon(prediction_df: pd.DataFrame, horizons: list[str], output_path: Path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Chance")

    prevalence_values = []
    for horizon in horizons:
        true_col = f"{horizon}_true"
        prob_col = f"{horizon}_prob"
        mask_col = f"{horizon}_mask"
        curve_df = prediction_df.loc[prediction_df[mask_col].eq(1), [true_col, prob_col]].dropna()
        if curve_df[true_col].nunique() < 2:
            continue

        y_true = curve_df[true_col].astype(int)
        y_prob = curve_df[prob_col].astype(float)
        prevalence_values.append(y_true.mean())

        fpr, tpr, _ = roc_curve(y_true, y_prob)
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
        horizon_label = HORIZON_LABELS[horizon]

        axes[0].plot(fpr, tpr, linewidth=2, label=f"{horizon_label} (AUROC {auroc:.3f})")
        axes[1].plot(recall, precision, linewidth=2, label=f"{horizon_label} (AUPRC {auprc:.3f})")

    if prevalence_values:
        axes[1].axhline(np.mean(prevalence_values), linestyle="--", color="gray", linewidth=1, label=f"Mean prevalence = {np.mean(prevalence_values):.3f}")

    axes[0].set_title("LightGBM multi-horizon: ROC curves by horizon")
    axes[0].set_xlabel("False positive rate")
    axes[0].set_ylabel("True positive rate")
    axes[0].set_xlim(0, 1)
    axes[0].set_ylim(0, 1)
    axes[0].grid(alpha=0.3)
    axes[0].legend(frameon=False, fontsize=9)

    axes[1].set_title("LightGBM multi-horizon: Precision-Recall curves by horizon")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[1].grid(alpha=0.3)
    axes[1].legend(frameon=False, fontsize=9)

    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()


plot_lgbm_multi_horizon_roc_pr_by_horizon(
    lgbm_multi_predictions,
    HORIZONS,
    output_path=FIGURE_DIR / "lgbm_multi_horizon_test_roc_pr_curves_by_horizon.png",
)


## Previous delirium stratified evaluation

In [ ]:

# Join previous-delirium feature at each anchor bin
anchor_prev_delirium = (
    binned[["stay_id", "bin", "prev_delirium"]]
    .rename(columns={"bin": "anchor_bin"})
)


def attach_prev_delirium_group(prediction_df: pd.DataFrame) -> pd.DataFrame:
    prediction_df = prediction_df.merge(anchor_prev_delirium, on=["stay_id", "anchor_bin"], how="left")
    prediction_df["prev_delirium_group"] = np.where(
        prediction_df["prev_delirium"].astype(float) >= 0.5,
        "prev_delirium",
        "no_prev_delirium",
    )
    return prediction_df


PREV_DELIRIUM_ORDER = ["previous no delirium", "previous delirium"]
PREV_DELIRIUM_LABELS = {
    "previous_no_delirium": "previous no delirium",
    "previous_delirium": "previous delirium",
}


def sort_prev_delirium_table(table: pd.DataFrame, extra_sort_cols: list[str] | None = None) -> pd.DataFrame:
    table = table.copy()
    table["comparison"] = pd.Categorical(table["comparison"], categories=PREV_DELIRIUM_ORDER, ordered=True)
    sort_cols = ["comparison", *(extra_sort_cols or [])]
    table = table.sort_values(sort_cols).reset_index(drop=True)
    table["comparison"] = table["comparison"].astype(str)
    return table


def stratified_metric_table(metrics_df: pd.DataFrame, label_map: dict[str, str]) -> pd.DataFrame:
    table = metrics_df.copy()
    table["comparison"] = table["analysis"].map(label_map).fillna(table["analysis"])
    cols = ["comparison", "model", *PREV_DELIRIUM_METRIC_COLUMNS, "n", "positive_rate"]
    table = table[cols].rename(columns=PREV_DELIRIUM_METRIC_RENAME)
    table = table.rename(columns={"positive_rate": "Positive_rate"})
    table = table[["comparison", "model", "Positive_rate", "AUROC", "AUPRC", "Sensitivity", "Specificity", "PPV", "NPV", "LR+", "LR-"]]
    return sort_prev_delirium_table(table)


within_predictions = attach_prev_delirium_group(within_predictions)
lgbm_multi_predictions = attach_prev_delirium_group(lgbm_multi_predictions)
mlp_multi_predictions = attach_prev_delirium_group(mlp_multi_predictions)
multi_predictions = attach_prev_delirium_group(multi_predictions)


In [ ]:

# LightGBM multi-horizon stratified evaluation by previous delirium
lgbm_prev_within_rows = []
for analysis, subset in [
    ("previous_no_delirium", lgbm_multi_predictions[lgbm_multi_predictions["prev_delirium_group"] == "no_prev_delirium"]),
    ("previous_delirium", lgbm_multi_predictions[lgbm_multi_predictions["prev_delirium_group"] == "prev_delirium"]),
]:
    add_metric_row(
        lgbm_prev_within_rows,
        analysis=analysis,
        model="LightGBM independent horizons",
        group=subset["prev_delirium_group"].iloc[0] if len(subset) else analysis,
        target="within_t_plus_2_delirium_from_horizons",
        y_true=subset["y_within_t_plus_2_true"],
        y_prob=subset["y_within_t_plus_2_prob_from_horizons"],
    )

lgbm_prev_within_metrics = pd.DataFrame(lgbm_prev_within_rows)
lgbm_prev_within_metrics.to_csv(MODELING_OUTPUT_DIR / "lgbm_multi_horizon_prev_delirium_stratified_metrics.csv", index=False)
lgbm_prev_within_display = stratified_metric_table(lgbm_prev_within_metrics, PREV_DELIRIUM_LABELS)
display(round_float_columns(lgbm_prev_within_display))

horizon_rows = []
for horizon in HORIZONS:
    true_col = f"{horizon}_true"
    mask_col = f"{horizon}_mask"
    prob_col = f"{horizon}_prob"
    for analysis, group_name in [
        ("previous_no_delirium", "no_prev_delirium"),
        ("previous_delirium", "prev_delirium"),
    ]:
        subset = lgbm_multi_predictions[
            (lgbm_multi_predictions["prev_delirium_group"] == group_name)
            & (lgbm_multi_predictions[mask_col].eq(1))
        ]
        add_metric_row(
            horizon_rows,
            analysis=analysis,
            model="LightGBM independent horizons",
            group=group_name,
            target=f"{horizon}_delirium",
            y_true=subset[true_col],
            y_prob=subset[prob_col],
        )

lgbm_prev_horizon_metrics = pd.DataFrame(horizon_rows)
lgbm_prev_horizon_metrics.to_csv(MODELING_OUTPUT_DIR / "lgbm_multi_horizon_prev_delirium_horizon_metrics.csv", index=False)
lgbm_prev_horizon_display = lgbm_prev_horizon_metrics.copy()
lgbm_prev_horizon_display["comparison"] = lgbm_prev_horizon_display["analysis"].map(PREV_DELIRIUM_LABELS)
lgbm_prev_horizon_display["horizon"] = (
    lgbm_prev_horizon_display["target"]
    .str.replace("_delirium", "", regex=False)
    .map(HORIZON_LABELS)
)
lgbm_prev_horizon_display = lgbm_prev_horizon_display[
    ["comparison", "horizon", *PREV_DELIRIUM_METRIC_COLUMNS, "n", "positive_rate"]
].rename(columns=PREV_DELIRIUM_METRIC_RENAME)
lgbm_prev_horizon_display = lgbm_prev_horizon_display.rename(columns={"positive_rate": "Positive_rate"})
lgbm_prev_horizon_display = lgbm_prev_horizon_display[
    ["comparison", "horizon", "Positive_rate", "AUROC", "AUPRC", "Sensitivity", "Specificity", "PPV", "NPV", "LR+", "LR-", "n"]
]
lgbm_prev_horizon_display["horizon"] = pd.Categorical(lgbm_prev_horizon_display["horizon"], categories=["t", "t+1", "t+2"], ordered=True)
lgbm_prev_horizon_display = sort_prev_delirium_table(lgbm_prev_horizon_display, extra_sort_cols=["horizon"])
display(round_float_columns(lgbm_prev_horizon_display))


## Save augmented prediction tables

In [ ]:

# Save row-level prediction tables with previous-delirium status
within_predictions.to_csv(MODELING_OUTPUT_DIR / "within_t_plus_2_test_predictions_all_models_with_prev_delirium.csv", index=False)
lgbm_multi_predictions.to_csv(MODELING_OUTPUT_DIR / "lgbm_multi_horizon_test_predictions_with_prev_delirium.csv", index=False)
mlp_multi_predictions.to_csv(MODELING_OUTPUT_DIR / "mlp_multi_horizon_test_predictions_with_prev_delirium.csv", index=False)
multi_predictions.to_csv(MODELING_OUTPUT_DIR / "lstm_gpu_test_predictions_with_prev_delirium.csv", index=False)

# Save key result tables separately
key_result_paths = [
    save_key_result(within_core_metrics, "within_t_plus_2_core_test_metrics.csv"),
    save_key_result(multi_core_metrics, "multi_horizon_core_test_metrics.csv"),
    save_key_result(lgbm_prev_within_display, "lgbm_prev_delirium_within_t_plus_2_metrics.csv"),
    save_key_result(lgbm_prev_horizon_display, "lgbm_prev_delirium_horizon_metrics.csv"),
    save_key_result(single_vs_multi_p_values_display, "single_vs_multi_horizon_within_t_plus_2_bootstrap_p_values.csv"),
]

best_within = within_core_metrics.sort_values("AUPRC", ascending=False).iloc[0]
best_multi_t2 = multi_core_metrics[multi_core_metrics["horizon"].eq("t+2")].sort_values("AUPRC", ascending=False).iloc[0]
summary_lines = [
    "# Key result summary",
    "",
    f"Best within t+2 model by AUPRC: {best_within['model']} "
    f"(AUROC {best_within['AUROC']:.3f}, AUPRC {best_within['AUPRC']:.3f})",
    f"Best multi-horizon model at t+2 by AUPRC: {best_multi_t2['model']} "
    f"(AUROC {best_multi_t2['AUROC']:.3f}, AUPRC {best_multi_t2['AUPRC']:.3f})",
    "",
    "Saved files:",
    *[f"- {path.name}" for path in key_result_paths],
    "- within_t_plus_2_bootstrap_metric_distributions.csv",
    "- single_vs_multi_horizon_within_t_plus_2_bootstrap_metric_distributions.csv",
]
(RESULT_ANALYSIS_DIR / "key_result_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")

print("Saved augmented prediction tables to", MODELING_OUTPUT_DIR)
print("Saved key result files to", RESULT_ANALYSIS_DIR)
